# Assesment Notebook

In [25]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [26]:
import numpy as np
from itertools import count

Problem 1: Representing SHA-256 Data

Before implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

Explain how you would represent:

32-bit words,
sequences of 32-bit words,
input messages,
512-bit message blocks,
the final 256-bit hash value,
big-endian integers.
For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

You can represent 32 bit words in python using an int. In sha-256 a 32 bit word is fundamentally a number. An int serves as a container to store the 32 bit word for the algorithms calculations. Python integers used to store words will be given a mask (0xFFFFFFFF) first to keep them in unsigned range.

In [27]:
# A 32-bit word represented as a standard Python integer (using hexadecimal)
word = 0x6a09e667
print(type(word))

<class 'int'>


A sequence of 32 bit words can be handled by extension with an list of ints. Same logic as an individual 32 bit word just stored in a list to keep them grouped. 

In [28]:
# Representing a sequence of 32-bit words as a Python list
word_sequence = [
    0x12345678,
    0xDEADBEEF,
    0xCAFEBABE,
    0x0BADF00D
]

print("Sequence type:", type(word_sequence))
print("Number of words:", len(word_sequence))
print("Second word in sequence:", hex(word_sequence[1]))

Sequence type: <class 'list'>
Number of words: 4
Second word in sequence: 0xdeadbeef


Input messages can be represented as bytes. SHA-256 will accept a sequence of bytes as an input, giving them a precise sequence to work with.

In [29]:
message = b"hello"
print(type(message))
print(len(message))

<class 'bytes'>
5


512-bit message blocks can be also be represented with bytes, exactly 64 as 512 bits equals 64 bytes.

In [30]:
message_block = bytes(64)
print(len(message_block))

64


the final 256-bit hash value can be represented with 32 bytes. Using similar logic to the 512 bit message block above 256 bits equals 32 bytes. 

In [31]:
hash_value = bytes(32)
print(len(hash_value))


32


big-endian integers can be represented using a specific int method python has. Using int.from_bytes() i can tell Python to interperet the first byte as the most significant byte. 

In [32]:
big_endian_data = bytes([0x12, 0x34, 0x56, 0x78])
number = int.from_bytes(big_endian_data, byteorder="big")

print(hex(number))

0x12345678


Problem 2: SHA-256 Bitwise Operations
Implement the Boolean and bitwise functions used by SHA-256:

Ch(x, y, z)
Maj(x, y, z)
Sigma0(x) 
Sigma1(x) 
sigma0(x)  
sigma1(x)  

Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

For each function:

provide an appropriate docstring,
explain the operation in Markdown,
implement the function directly from that definition,
provide tests on appropriate inputs.


    SHA-256 Choose function.
    
    Parameters:
    x, y, z (array_like): 32-bit unsigned integers.
    
    Returns:
    numpy.ndarray or numpy.uint32: Result of (x AND y) XOR ((NOT x) AND z).

    Explanations: 
    This function chooses bit from y or z depending on the value of x for each position. If x is 0 it chooses y and if it x is 1 it chooses z.  
    

In [33]:
def ch(x, y, z):
    
    x = np.asarray(x, dtype=np.uint32)
    y = np.asarray(y, dtype=np.uint32)
    z = np.asarray(z, dtype=np.uint32)
    return (x & y) ^ (~x & z)

In [34]:
# Test Ch with simple bitmasks
x_val = np.uint32(0x0000FFFF)
y_val = np.uint32(0xFFFFFFFF)
z_val = np.uint32(0x00000000)
print(f"Ch test: {hex(ch(x_val, y_val, z_val))}")  # Expected: 0xffff0000

Ch test: 0xffff


 
    SHA-256 Majority function.
    
    Parameters:
    x, y, z (array_like): 32-bit unsigned integers.
    
    Returns:
    numpy.ndarray or numpy.uint32: Result of (x AND y) XOR (x AND z) XOR (y AND z).

    Explanation:
    The Majority function sets a bit set to 1 if at least 2 of the 3 input bits were 1. 
    

In [35]:
def maj(x, y, z):
   
    x = np.asarray(x, dtype=np.uint32)
    y = np.asarray(y, dtype=np.uint32)
    z = np.asarray(z, dtype=np.uint32)
    return (x & y) ^ (x & z) ^ (y & z)

In [36]:
x_val = np.uint32(0b1100)
y_val = np.uint32(0b1010)
z_val = np.uint32(0b1100)
print(f"Maj test: {bin(maj(x_val, y_val, z_val))}")  # Expected: 0b1100

Maj test: 0b1100


For the next implementation I am going to make a helper function. The Sigma0 methods use a bitwise operation known as ROTR (Right Rotate) to shift bits to the right. Any bits that are shifted out of the 8 bits wrap back around to the left side. 

In [37]:
def _rotr(x, n):
    """Circular right rotation of a 32-bit unsigned integer by n bits."""
    x = np.asarray(x, dtype=np.uint32)
    return np.bitwise_or(x >> n, x << (32 - n))

SHA-256 Big Sigma 0 compression function.
    
    Parameters:
    x (array_like): 32-bit unsigned integer.
    
    Returns:
    numpy.ndarray or numpy.uint32: ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x).

    Explanation:
    Sigma 0 is used in the compression loop to mix bits of working variables in each of the rounds of sha 256. This is done to ensure that any change to the input completely changes the hash.

In [38]:
def Sigma_0(x):
    x = np.asarray(x, dtype=np.uint32)
    return _rotr(x, 2) ^ _rotr(x, 13) ^ _rotr(x, 22)

In [39]:
x_val = np.uint32(0x12345678)
print(f"Sigma0 test: {hex(Sigma_0(x_val))}")

Sigma0 test: 0x66146474


SHA-256 Big Sigma 1 compression function.
    
    Parameters:
    x (array_like): 32-bit unsigned integer.
    
    Returns:
    numpy.ndarray or numpy.uint32: ROTR^6(x) XOR ROTR^11(x) XOR ROTR^25(x).

    Explanation:
    Much like Sigma0, Sigma1 works very similarly it just rotates the bits in a different amount and works on a different set of variables. This is done to prevent symmetry in the hash.  

In [40]:
def Sigma_1(x):
    x = np.asarray(x, dtype=np.uint32)
    return _rotr(x, 6) ^ _rotr(x, 11) ^ _rotr(x, 25)

In [41]:
x_val = np.uint32(0x12345678)
print(f"Sigma1 test: {hex(Sigma_1(x_val))}")

Sigma1 test: 0x3561abda


The small s sigma functions work similarly to the big s functions however they use a secondary function alongside rotr called SHR (Shift Right). SHR as the name suggests shifts bits to the right however unlike rotr the bits are not looped back to the beginning and instead drop off the right as the left fill with an equivalent amount of 0s.

In [42]:
def shr(x, n):
    shifted = x >> n
    return shifted & 0xFFFFFFFF


SHA-256 Small Sigma 0 message schedule function.
    
    Parameters:
    x (array_like): 32-bit unsigned integer.
    
    Returns:
    numpy.ndarray or numpy.uint32: ROTR^7(x) XOR ROTR^18(x) XOR SHR^3(x).
    
    Explanation:
    The small sigma functions are used outside the compression loop to expand the message schedule. While the first 16 words are taken directly from the input message, sigma0 and sigma1 take those earlier words, scramble them, and use them to generate the remaining words until the schedule fills all 64 words needed for the compression loop.

In [43]:
def sigma0(x):
    return _rotr(x, 7) ^ _rotr(x, 18) ^ shr(x, 3);

In [44]:
x_val = np.uint32(0x12345678)
print(f"small sigma0 test: {hex(sigma0(x_val))}")

small sigma0 test: 0xe7fce6ee


SHA-256 Small Sigma 1 message schedule function.
    
    Parameters:
    x (array_like): 32-bit unsigned integer.
    
    Returns:
    numpy.ndarray or numpy.uint32: ROTR^17(x) XOR ROTR^19(x) XOR SHR^10(x).

In [45]:
def sigma1(x):
    return _rotr(x, 17) ^ _rotr(x, 19) ^ shr(x, 10);

In [46]:
x_val = np.uint32(0x12345678)
print(f"small sigma1 test: {hex(sigma1(x_val))}")

small sigma1 test: 0xa1f78649


Problem 3: Generating the SHA-256 Constants
The Secure Hash Standard defines the initial hash values and round constants used by SHA-256 using fractional parts of square and cube roots of prime numbers.

Use Python and NumPy to generate the 64 round constants specified in the standard.

Your solution should:

1. write a function primes(n) that generates the first n prime numbers,
2. calculate the cube roots of the first 64 primes,
3. extract the fractional part of each cube root,
4. convert the resulting to a 32-bit unsigned integer,
5. display the resulting constants in hexadecimal,
6. compare your results with the constants in the Secure Hash Standard.

In [ ]:
def isprime(i):
            """Test whether n is prime."""
            # Loop through 2...n-1.
            for j in range(2, i):
                # Calculate remainder of i divided by j.
                if i % j == 0:
                    # If this is zero, then i is not prime.
                    return False
            # If we get here, then i is prime.
            return True
#This is from prime-number.ipynb in the course learning materials
        

In [48]:
def primes(n):
    primesList = []
    for i in count(2):
        if isprime(i):
            primesList.append(i)
            yield i
            if len(primesList) >= n:
                        break
        

In [51]:
print(list(primes(100)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503, 509, 521, 523, 541]
